In [19]:
# !pip3 install fairlearn

In [20]:
# Notebook based on https://fairlearn.org/v0.13/quickstart.html

In [21]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fairlearn.datasets import fetch_diabetes_hospital
data = fetch_diabetes_hospital(as_frame=True)
X = data.data.copy()
X.drop(columns=["readmitted", "readmit_binary"], inplace=True)
y = data.target
X_ohe = pd.get_dummies(X)
sensitiv = X['race']


In [22]:
X.columns
# race.value_counts()

Index(['race', 'gender', 'age', 'discharge_disposition_id',
       'admission_source_id', 'time_in_hospital', 'medical_specialty',
       'num_lab_procedures', 'num_procedures', 'num_medications',
       'primary_diagnosis', 'number_diagnoses', 'max_glu_serum', 'A1Cresult',
       'insulin', 'change', 'diabetesMed', 'medicare', 'medicaid',
       'had_emergency', 'had_inpatient_days', 'had_outpatient_days'],
      dtype='str')

## Basic stats about predictions

In [23]:
from fairlearn.metrics import MetricFrame, selection_rate
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
np.random.seed(42)  # set seed for consistent results
X_train, X_test, y_train, y_test, A_train, A_test = train_test_split(X_ohe, y, sensitiv, random_state=123)
classifier = DecisionTreeClassifier(min_samples_leaf=10, max_depth=4)
classifier.fit(X_train, y_train)

# threshold since we get probabilities predicted
y_pred = (classifier.predict_proba(X_test)[:,1] >= 0.1)

In [24]:
# accuracy per group
mf_acc = MetricFrame(metrics=accuracy_score, y_true=y_test, y_pred=y_pred, sensitive_features=A_test)
print("overall accuracy = ",mf_acc.overall.item())
print("by group: ")
print(mf_acc.by_group)

overall accuracy =  0.5148573225375364
by group: 
race
AfricanAmerican    0.530935
Asian              0.658683
Caucasian          0.503535
Hispanic           0.612524
Other              0.591549
Unknown            0.574576
Name: accuracy_score, dtype: float64


In [25]:
# selection rate (percentage predicted in the positive class)
mf = MetricFrame(metrics=selection_rate, y_true=y_test, y_pred=y_pred, sensitive_features=A_test)
print("overall selection rate = ",mf.overall.item())
print("by group: ")
print(mf.by_group)

overall selection rate =  0.5275528653407751
by group: 
race
AfricanAmerican    0.514080
Asian              0.341317
Caucasian          0.539675
Hispanic           0.426614
Other              0.445070
Unknown            0.438983
Name: selection_rate, dtype: float64


In [26]:
# FPR (false positive rate)
from fairlearn.metrics import false_positive_rate
mf_fpr = MetricFrame(metrics=false_positive_rate, y_true=y_test, y_pred=y_pred, sensitive_features=A_test)
print("overall FPR = ",mf_fpr.overall.item())
print("by group: ")
print(mf_fpr.by_group)

overall FPR =  0.5071425412409889
by group: 
race
AfricanAmerican    0.490527
Asian              0.326797
Caucasian          0.520369
Hispanic           0.396514
Other              0.417722
Unknown            0.425926
Name: false_positive_rate, dtype: float64


In [27]:
# TPR (true positive rate)
from fairlearn.metrics import true_positive_rate
mf_tpr = MetricFrame(metrics=true_positive_rate, y_true=y_test, y_pred=y_pred, sensitive_features=A_test)
print("overall TPR = ",mf_tpr.overall.item())
print("by group: ")
print(mf_tpr.by_group)

overall TPR =  0.6905687036382904
by group: 
race
AfricanAmerican    0.703911
Asian              0.500000
Caucasian          0.691445
Hispanic           0.692308
Other              0.666667
Unknown            0.580000
Name: true_positive_rate, dtype: float64


## Evaluating fairness

In [28]:
# Demographic parity = all groups should predict the positive class equally (i.e. equal selection rate))
from fairlearn.metrics import demographic_parity_difference
from fairlearn.metrics import demographic_parity_ratio

demog_parity_diff = demographic_parity_difference(y_test,
                                    y_pred,
                                    sensitive_features=A_test)
demog_parity_ratio = demographic_parity_ratio(y_test,
                               y_pred,
                               sensitive_features=A_test)

print("Demographic parity difference = ",demog_parity_diff)
print("Demographic parity ratio = ",demog_parity_ratio)


Demographic parity difference =  0.19835763736850454
Demographic parity ratio =  0.6324498329570207


## Mitigations (in-processing)

In [29]:
from fairlearn.reductions import ErrorRate, EqualizedOdds, ExponentiatedGradient
objective = ErrorRate(costs={'fp': 0.1, 'fn': 0.9})
constraint = EqualizedOdds(difference_bound=0.01)

classifier = DecisionTreeClassifier(min_samples_leaf=10, max_depth=4)

mitigator = ExponentiatedGradient(classifier, constraint, objective=objective)
mitigator.fit(X_train, y_train, sensitive_features=A_train)

y_pred_mitigated = mitigator.predict(X_test)

In [30]:
# accuracy per group
mf_acc = MetricFrame(metrics=accuracy_score, y_true=y_test, y_pred=y_pred_mitigated, sensitive_features=A_test)
print("overall accuracy = ",mf_acc.overall.item())
print("by group: ")
print(mf_acc.by_group)

overall accuracy =  0.5251159500039305
by group: 
race
AfricanAmerican    0.524358
Asian              0.562874
Caucasian          0.525588
Hispanic           0.549902
Other              0.478873
Unknown            0.511864
Name: accuracy_score, dtype: float64


In [31]:
# selection rate (percentage predicted in the positive class)
mf = MetricFrame(metrics=selection_rate, y_true=y_test, y_pred=y_pred_mitigated, sensitive_features=A_test)
print("overall selection rate = ",mf.overall.item())
print("by group: ")
print(mf.by_group)

overall selection rate =  0.5055027120509394
by group: 
race
AfricanAmerican    0.508736
Asian              0.473054
Caucasian          0.504643
Hispanic           0.465753
Other              0.569014
Unknown            0.511864
Name: selection_rate, dtype: float64


In [32]:
# FPR (false positive rate)
from fairlearn.metrics import false_positive_rate
mf_fpr = MetricFrame(metrics=false_positive_rate, y_true=y_test, y_pred=y_pred_mitigated, sensitive_features=A_test)
print("overall FPR = ",mf_fpr.overall.item())
print("by group: ")
print(mf_fpr.by_group)

overall FPR =  0.4889655477422494
by group: 
race
AfricanAmerican    0.491220
Asian              0.450980
Caucasian          0.488195
Hispanic           0.453159
Other              0.550633
Unknown            0.500000
Name: false_positive_rate, dtype: float64


In [33]:
# TPR (true positive rate)
from fairlearn.metrics import true_positive_rate
mf_tpr = MetricFrame(metrics=true_positive_rate, y_true=y_test, y_pred=y_pred_mitigated, sensitive_features=A_test)
print("overall TPR = ",mf_tpr.overall.item())
print("by group: ")
print(mf_tpr.by_group)

overall TPR =  0.6375838926174496
by group: 
race
AfricanAmerican    0.649907
Asian              0.714286
Caucasian          0.633941
Hispanic           0.576923
Other              0.717949
Unknown            0.640000
Name: true_positive_rate, dtype: float64


In [34]:
# Demographic parity = all groups should predict the positive class equally (i.e. equal selection rate))
from fairlearn.metrics import demographic_parity_difference
from fairlearn.metrics import demographic_parity_ratio

demog_parity_diff = demographic_parity_difference(y_test,
                                    y_pred_mitigated,
                                    sensitive_features=A_test)
demog_parity_ratio = demographic_parity_ratio(y_test,
                               y_pred_mitigated,
                               sensitive_features=A_test)

print("Demographic parity difference = ",demog_parity_diff)
print("Demographic parity ratio = ",demog_parity_ratio)


Demographic parity difference =  0.10326065984950805
Demographic parity ratio =  0.8185270581852705


In [35]:
# Demographic parity = all groups should predict the positive class equally (i.e. equal selection rate))
from fairlearn.metrics import equalized_odds_difference
from fairlearn.metrics import equalized_odds_ratio

equal_odds_diff = equalized_odds_difference(y_test,
                                    y_pred_mitigated,
                                    sensitive_features=A_test)
equal_odds_ratio = equalized_odds_ratio(y_test,
                               y_pred_mitigated,
                               sensitive_features=A_test)

print("Equalized odds difference = ",equal_odds_diff)
print("equalized odds ration ratio = ",equal_odds_ratio)


Equalized odds difference =  0.14102564102564108
equalized odds ration ratio =  0.8035714285714285


# Equalized odds

The order is a bit weird because I already added this cell below the mitigation cells, oops.

In [36]:
# Demographic parity = all groups should predict the positive class equally (i.e. equal selection rate))
from fairlearn.metrics import equalized_odds_difference
from fairlearn.metrics import equalized_odds_ratio

equal_odds_diff = equalized_odds_difference(y_test,
                                    y_pred,
                                    sensitive_features=A_test)
equal_odds_ratio = equalized_odds_ratio(y_test,
                               y_pred,
                               sensitive_features=A_test)

print("Equalized odds difference = ",equal_odds_diff)
print("equalized odds ration ratio = ",equal_odds_ratio)


Equalized odds difference =  0.2039106145251397
equalized odds ration ratio =  0.6280112044817927
